# Zero-Shot FFB 3D Perception Pipeline
**YOLO-World → SAM 2 → 2.5D Grid Integration → Mass via 956.28 kg/m³**

Batch / offline — top-15 frames per bundle, IQR outlier rejection, median aggregation, 2×T4 parallel.

In [ ]:
# ── 0. Pre-flight ─────────────────────────────────────────────────────────
import os, sys, pathlib

PROJECT_DIR = '/kaggle/input/datasets/rajulkabir/rgbd-mass-ffb'

for candidate in [
    f'{PROJECT_DIR}/data',
    f'{PROJECT_DIR}/data/data',
    PROJECT_DIR,
]:
    subdirs = [d for d in os.listdir(candidate)
               if os.path.isdir(os.path.join(candidate, d)) and d.startswith('FFB')]
    if subdirs:
        DATA_DIR = candidate
        break
else:
    raise RuntimeError('Cannot find FFB folders under ' + PROJECT_DIR)

sys.path.insert(0, PROJECT_DIR)

for f in ('perception_pipeline.py', 'bag_reader.py', 'ground_truth.csv', 'validation_density.csv'):
    print(f'  {"OK " if (pathlib.Path(PROJECT_DIR)/f).exists() else "MISSING"}  {f}')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

bundles = sorted(d for d in os.listdir(DATA_DIR)
                 if os.path.isdir(os.path.join(DATA_DIR, d)) and d.startswith('FFB'))
print(f'DATA_DIR : {DATA_DIR}')
print(f'Bundles  : {bundles}')
print(f'Device   : {DEVICE}  ({N_GPUS} GPU(s))')
if torch.cuda.is_available():
    for i in range(N_GPUS):
        p = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')

In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────
import subprocess

def pip(*args, check=True):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args],
                       capture_output=True, text=True)
    if check and r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
        raise RuntimeError(f'pip failed: {args}')
    return r.returncode == 0

def git_clone(url, dest):
    if os.path.exists(dest):
        print(f'  already present: {dest}'); return
    r = subprocess.run(['git', 'clone', '--depth=1', url, dest],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:]); raise RuntimeError(f'git clone failed: {url}')

pip('ultralytics>=8.2');           print('ultralytics OK')

sam2_ok = (
    pip('sam2', check=False) or
    pip('git+https://github.com/facebookresearch/segment-anything-2.git@v1.0', check=False) or
    pip('git+https://github.com/facebookresearch/segment-anything-2.git@main', check=False) or
    pip('git+https://github.com/facebookresearch/segment-anything-2.git', check=False)
)
if not sam2_ok: raise RuntimeError('SAM2 install failed')
print('SAM2 OK')

git_clone('https://github.com/DepthAnything/Depth-Anything-V2.git', 'Depth-Anything-V2')
if 'Depth-Anything-V2' not in sys.path: sys.path.insert(0, 'Depth-Anything-V2')
print('DepthAnything V2 OK')

pip('pyrealsense2');               print('pyrealsense2 OK')
pip('open3d');                     print('open3d OK')
print('\nAll dependencies installed.')

In [ ]:
# ── 2. Download model weights ──────────────────────────────────────────────
import urllib.request, shutil, pathlib

# CLIP is loaded internally by YOLO-World. Delete any corrupted cache so it
# re-downloads cleanly (SHA256 mismatch from a previous partial download).
_clip_cache = pathlib.Path.home() / '.cache' / 'clip'
if _clip_cache.exists():
    shutil.rmtree(_clip_cache)
    print(f'  cleared CLIP cache: {_clip_cache}')

WEIGHTS = {
    'sam2_hiera_small.pt':        'https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt',
    'yolov8s-world.pt':           'https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8s-world.pt',
    'yolov8m-world.pt':           'https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8m-world.pt',
    'depth_anything_v2_vits.pth': 'https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth',
}
for fname, url in WEIGHTS.items():
    if os.path.exists(fname):
        print(f'  cached: {fname}')
    else:
        print(f'  downloading {fname} ...')
        urllib.request.urlretrieve(url, fname)
        print(f'  done ({os.path.getsize(fname)/1e6:.1f} MB)')

In [ ]:
# ── 3. Imports, patches, ground truth, density constant ───────────────────
from perception_pipeline import (
    FFBPerceptionPipeline, CameraIntrinsics, PerceptionResult,
    BundleResult, process_bundle_multi_frame,
)
import perception_pipeline as _pp
from bag_reader import find_bundle_bags, get_depth_scale, get_intrinsics_from_bag
import numpy as np, cv2, glob as _glob
import matplotlib.pyplot as plt
import pandas as pd


# ── Patch _load_sam2 ──────────────────────────────────────────────────────
def _patched_load_sam2(config, checkpoint, device='cpu'):
    import torch._jit_internal as _jit_int
    if not getattr(_jit_int, '_clear_fn_overloads_patched', False):
        _orig_clear = _jit_int._clear_fn_overloads
        def _safe_clear_fn_overloads(qual_name):
            try: _orig_clear(qual_name)
            except KeyError: pass
        _jit_int._clear_fn_overloads = _safe_clear_fn_overloads
        _jit_int._clear_fn_overloads_patched = True

    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    bn  = os.path.basename(config)
    bne = os.path.splitext(bn)[0]
    candidates = dict.fromkeys([
        config, bn, bne,
        bn .replace('sam2_', 'sam2.1_'), bne.replace('sam2_', 'sam2.1_'),
        bn .replace('sam2.1_', 'sam2_'), bne.replace('sam2.1_', 'sam2_'),
    ])
    import sam2 as _sam2mod
    disk = _glob.glob(os.path.join(os.path.dirname(_sam2mod.__file__), '**', '*.yaml'), recursive=True)
    for d in disk:
        candidates[d] = candidates[os.path.basename(d)] = candidates[os.path.splitext(os.path.basename(d))[0]] = None
    for cfg in candidates:
        try:
            model = build_sam2(cfg, checkpoint, device=device)
            print(f'  SAM2 loaded with config: {cfg}')
            break
        except Exception:
            pass
    else:
        raise RuntimeError(f'SAM2: no working config found. Disk configs: {disk}')
    pred = SAM2ImagePredictor(model)
    if device == 'cpu':
        pred.model = pred.model.float()
    return pred

_pp._load_sam2 = _patched_load_sam2


# ── Patch _detect ─────────────────────────────────────────────────────────
def _patched_detect(self, rgb: np.ndarray):
    import torch as _torch
    h, w = rgb.shape[:2]
    with _torch.inference_mode():
        results = self.yolo.predict(rgb, verbose=False, conf=0.01)
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        margin_x, margin_y = w * 0.25, h * 0.25
        return [margin_x, margin_y, w - margin_x, h - margin_y], 0.0
    confidences = boxes.conf.cpu().numpy()
    best = int(np.argmax(confidences))
    return boxes.xyxy[best].cpu().numpy().tolist(), float(confidences[best])

FFBPerceptionPipeline._detect = _patched_detect


# ── Depth segmentation helpers ────────────────────────────────────────────
def _auto_margin(depth_m, z_front, valid,
                 search_lo=0.05, search_hi=0.45, n_bins=80, fallback_m=0.10):
    band = depth_m[valid & (depth_m > z_front + search_lo)
                        & (depth_m < z_front + search_hi)]
    if band.size < 200:
        return fallback_m
    hist, edges = np.histogram(band, bins=n_bins)
    z_tarp = float(edges[int(np.argmax(hist))])
    return float(np.clip(z_tarp - z_front - 0.05, 0.08, 0.22))


def _depth_foreground_mask(depth_m, rgb=None,
                           min_area_frac=0.003, max_area_frac=0.35,
                           colour_s_min=40, colour_v_max=160):
    H, W  = depth_m.shape
    valid = (depth_m > 0.1) & (depth_m < 10.0)
    if not valid.any():
        return None, None, None

    z_front  = float(np.percentile(depth_m[valid], 5))
    margin_m = _auto_margin(depth_m, z_front, valid)
    fg = (valid & (depth_m <= z_front + margin_m)).astype(np.uint8)

    if rgb is not None:
        rgb_d = cv2.resize(rgb, (W, H), interpolation=cv2.INTER_LINEAR) if rgb.shape[:2] != (H, W) else rgb
        hsv   = cv2.cvtColor(rgb_d, cv2.COLOR_RGB2HSV)
        fg    = (fg.astype(bool)
                 & (hsv[..., 1] >= colour_s_min)
                 & (hsv[..., 2] <= colour_v_max)).astype(np.uint8)

    k  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    fg = cv2.morphologyEx(cv2.morphologyEx(fg, cv2.MORPH_CLOSE, k), cv2.MORPH_OPEN, k)

    contours, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, None, None

    img_area       = H * W
    cx_img, cy_img = W * 0.5, H * 0.5
    half_diag      = (cx_img**2 + cy_img**2) ** 0.5

    def _dist_centre(cnt):
        M = cv2.moments(cnt)
        if M['m00'] == 0: return float('inf')
        return ((M['m10']/M['m00'] - cx_img)**2 + (M['m01']/M['m00'] - cy_img)**2) ** 0.5

    sized = [c for c in contours
             if min_area_frac * img_area < cv2.contourArea(c) < max_area_frac * img_area]
    if not sized:
        return None, None, None

    c = min(sized, key=_dist_centre)
    if _dist_centre(c) > 0.75 * half_diag:
        c = max(sized, key=cv2.contourArea)
        if _dist_centre(c) > 0.75 * half_diag:
            return None, None, None

    mask = np.zeros((H, W), dtype=np.uint8)
    cv2.drawContours(mask, [c], -1, 1, thickness=cv2.FILLED)
    if mask.mean() > max_area_frac:
        return None, None, None

    x, y, w, h = cv2.boundingRect(c)
    return mask.astype(bool), [float(x), float(y), float(x+w), float(y+h)], margin_m


def _expand_mask_bbox(fg_mask, depth_m, rgb_image, z_front,
                      z_window=0.25, s_min=25, v_max=160, pad_px=20):
    dh, dw = depth_m.shape
    if not fg_mask.any():
        return fg_mask

    ys, xs = np.where(fg_mask)
    y1 = max(0,  ys.min() - pad_px);  y2 = min(dh, ys.max() + pad_px)
    x1 = max(0,  xs.min() - pad_px);  x2 = min(dw, xs.max() + pad_px)
    search = np.zeros((dh, dw), dtype=bool)
    search[y1:y2, x1:x2] = True

    depth_ok = (depth_m > 0) & (depth_m <= z_front + z_window)

    if rgb_image is not None:
        rh, rw = rgb_image.shape[:2]
        rgb_d  = cv2.resize(rgb_image, (dw, dh), interpolation=cv2.INTER_LINEAR) if (rh, rw) != (dh, dw) else rgb_image
        hsv       = cv2.cvtColor(rgb_d, cv2.COLOR_RGB2HSV)
        colour_ok = (hsv[..., 1] >= s_min) & (hsv[..., 2] <= v_max)
    else:
        colour_ok = np.ones((dh, dw), dtype=bool)

    expanded = search & depth_ok & colour_ok
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    expanded = cv2.morphologyEx(expanded.astype(np.uint8), cv2.MORPH_OPEN, k).astype(bool)
    return expanded if expanded.any() else fg_mask


# ── Volume estimator (2.5D grid integration only) ─────────────────────────
SCALE_2D5 = 2.02
SCALE_BY_WIDTH = {1280: 2.02, 848: 2.53}


def _compute_2d5_volume(point_cloud: np.ndarray, grid_step: float = 0.002) -> tuple:
    if point_cloud.shape[0] < 20:
        return 0.0, 0.0
    x, y, z = point_cloud.T.astype(np.float64)
    z_ref    = float(np.percentile(z, 95))
    x0, y0  = float(x.min()), float(y.min())
    xi = np.floor((x - x0) / grid_step).astype(np.int32)
    yi = np.floor((y - y0) / grid_step).astype(np.int32)
    ny = int(yi.max()) + 1
    z_top = np.full((int(xi.max())+1) * ny, z_ref, dtype=np.float64)
    np.minimum.at(z_top, xi * ny + yi, z)
    has_data = z_top < (z_ref - 1e-4)
    return float(np.sum(z_ref - z_top[has_data]) * grid_step**2), z_ref


def _patched_compute_volume(point_cloud: np.ndarray, scale: float = SCALE_2D5) -> tuple:
    if point_cloud.shape[0] < 4:
        return 0.0, 0.0, 0.0
    v_2d5, z_ref = _compute_2d5_volume(point_cloud)
    if v_2d5 > 0:
        return v_2d5 * scale, v_2d5, z_ref
    return 0.0, 0.0, 0.0

FFBPerceptionPipeline._compute_convex_volume = staticmethod(
    lambda pc: _patched_compute_volume(pc)[0])


# ── Patch process_frame ───────────────────────────────────────────────────
def _patched_process_frame(self, rgb_image, depth_map, camera_intrinsics):
    depth_m = FFBPerceptionPipeline._to_metric_depth(depth_map, camera_intrinsics.depth_scale)
    depth_m = cv2.medianBlur(depth_m, 3)
    dh, dw  = depth_m.shape
    rh, rw  = rgb_image.shape[:2]

    cam_width = int(getattr(camera_intrinsics, 'width', 0))
    scale     = SCALE_BY_WIDTH.get(cam_width, SCALE_2D5)

    fg_mask, _, _ = _depth_foreground_mask(depth_m, rgb=rgb_image)

    if fg_mask is not None:
        z_front = float(np.percentile(depth_m[depth_m > 0.1], 5))
        if cam_width == 848:
            mask_depth_res = _expand_mask_bbox(fg_mask, depth_m, rgb_image, z_front)
            expand_tag = 'expanded'
        else:
            mask_depth_res = fg_mask
            expand_tag = 'tight'
        confidence = 0.95
        print(f'    [depth]  z_front={z_front:.3f}m  mask={mask_depth_res.mean()*100:.1f}% ({expand_tag})  scale={scale}')
    else:
        bbox_yolo, confidence = FFBPerceptionPipeline._detect(self, rgb_image)
        x1, y1, x2, y2 = bbox_yolo
        if (x2-x1)*(y2-y1) / (rh*rw) > 0.40:
            print('    [yolo-skip]  box too large — zero frame')
            return PerceptionResult(
                mask=np.zeros((dh, dw), dtype=bool),
                point_cloud=np.zeros((0, 3), dtype=np.float32),
                estimated_convex_volume=0.0,
                detection_confidence=0.0,
                depth_fill_ratio=0.0,
            )
        print(f'    [yolo-sam2]  bbox={[int(v) for v in bbox_yolo]}  conf={confidence:.2f}')
        mask_rgb = self._segment(rgb_image, bbox_yolo)
        mask_depth_res = (cv2.resize(mask_rgb.astype(np.uint8), (dw, dh),
                                     interpolation=cv2.INTER_NEAREST).astype(bool)
                          if (rh, rw) != (dh, dw) else mask_rgb)
        z_front = float(np.percentile(depth_m[depth_m > 0.1], 5))

    fill_ratio = 0.0
    if self.use_depth_fusion and self.depth_model is not None:
        rgb_fuse   = cv2.resize(rgb_image, (dw, dh)) if (rh, rw) != (dh, dw) else rgb_image
        depth_m, fill_ratio = self._fuse_depth(rgb_fuse, depth_m, mask_depth_res)

    point_cloud    = self._project_to_3d(depth_m, mask_depth_res, camera_intrinsics)
    v_2d5_raw, z_ref = _compute_2d5_volume(point_cloud)
    volume         = v_2d5_raw * scale if v_2d5_raw > 0 else 0.0

    print(f'    [vol]  2d5_raw={v_2d5_raw*1000:.2f}L  ×{scale:.2f}={volume*1000:.2f}L  '
          f'dome={( z_ref - z_front)*100:.1f}cm')

    return PerceptionResult(
        mask=mask_depth_res,
        point_cloud=point_cloud,
        estimated_convex_volume=volume,
        detection_confidence=float(confidence),
        depth_fill_ratio=float(fill_ratio),
    )

FFBPerceptionPipeline.process_frame = _patched_process_frame


# ── Patch _project_to_3d ──────────────────────────────────────────────────
@staticmethod
def _patched_project_to_3d(depth_m, mask, K):
    h, w   = depth_m.shape
    us, vs = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
    valid  = mask & (depth_m > 0)
    Z = depth_m[valid].astype(np.float32)
    u = us[valid]; v = vs[valid]
    if Z.size > 0:
        keep    = Z <= float(np.percentile(Z, 5)) + 0.25
        Z, u, v = Z[keep], u[keep], v[keep]
    X = (u - K.cx) * Z / K.fx
    Y = (v - K.cy) * Z / K.fy
    return np.stack([X, Y, Z], axis=-1)

FFBPerceptionPipeline._project_to_3d = _patched_project_to_3d


# ── Patch _fuse_depth (depth model device routing + multiple-of-14 resize) ─
def _patched_fuse_depth(self, rgb, depth_m, mask):
    import torchvision.transforms.functional as TF
    import torch as _torch

    void_mask = mask & (depth_m <= 0)
    n_void = int(void_mask.sum()); n_mask = int(mask.sum())
    if n_void == 0 or n_mask == 0:
        return depth_m, 0.0

    h, w = rgb.shape[:2]
    h14  = ((h + 13) // 14) * 14
    w14  = ((w + 13) // 14) * 14
    rgb_in = cv2.resize(rgb, (w14, h14)) if (h14, w14) != (h, w) else rgb

    dev = next(self.depth_model.parameters()).device
    with _torch.inference_mode():
        rel_depth = self.depth_model(TF.to_tensor(rgb_in).unsqueeze(0).to(dev))
    rel_depth = rel_depth.squeeze().cpu().numpy().astype(np.float32)
    if (h14, w14) != (h, w):
        rel_depth = cv2.resize(rel_depth, (w, h), interpolation=cv2.INTER_LINEAR)

    valid = mask & (depth_m > 0)
    if valid.sum() < 10:
        return depth_m, 0.0

    A = np.stack([rel_depth[valid], np.ones(valid.sum())], axis=1)
    sc, sh = np.linalg.lstsq(A, depth_m[valid], rcond=None)[0]
    fused  = depth_m.copy()
    fused[void_mask] = np.clip(rel_depth[void_mask] * sc + sh, 0, None)
    return fused, float(n_void / n_mask)

FFBPerceptionPipeline._fuse_depth = _patched_fuse_depth

print('All pipeline patches applied.')


# ── Ground truth & density constant ──────────────────────────────────────
GT = pd.read_csv(f'{PROJECT_DIR}/ground_truth.csv')
GT.index = GT['FFB_No'].astype(int)

def gt(ffb_name):
    num = int(ffb_name.replace('FFB', ''))
    return GT.loc[num].to_dict() if num in GT.index else {}

DENSITY_CONSTANT = GT['True_Density_kg_L'].mean() * 1000
print(f'Density constant: {DENSITY_CONSTANT:.2f} kg/m³')

val = pd.read_csv(f'{PROJECT_DIR}/validation_density.csv')
val['pred_mass_kg'] = DENSITY_CONSTANT * val['Estimated_Volume_m3']
val['abs_err']      = (val['pred_mass_kg'] - val['Target_Mass_kg']).abs()
val['pct_err']      = 100 * val['abs_err'] / val['Target_Mass_kg']
print(f'Density constant validation (n=10):  '
      f'MAE={val["abs_err"].mean():.2f} kg  MAPE={val["pct_err"].mean():.1f}%')

In [ ]:
# ── 3c. Patch process_bundle_multi_frame: ranked frame selection + IQR ────
import perception_pipeline as _pp_mod

def _patched_process_bundle(ffb_dir, pipeline, camera_intrinsics,
                            n_frames=5, max_scan=60):
    from bag_reader import iter_bundle

    # rank frames by valid depth count in centre crop
    candidates = []
    for rgb, depth in iter_bundle(ffb_dir, max_frames=max_scan):
        h, w  = depth.shape
        crop  = depth[h//4: 3*h//4, w//4: 3*w//4]
        candidates.append((int((crop > 0).sum()), rgb.copy(), depth.copy()))
    candidates.sort(key=lambda x: x[0], reverse=True)

    frame_data = []  # (depth_score, volume, result)
    for score, rgb, depth in candidates[:min(n_frames, len(candidates))]:
        result = pipeline.process_frame(rgb, depth, camera_intrinsics)
        frame_data.append((score, result.estimated_convex_volume, result))

    if not frame_data:
        return _pp_mod.BundleResult(
            median_volume=0.0, std_volume=0.0, mean_volume=0.0,
            merged_volume=0.0, per_frame_volumes=[],
            best_point_cloud=np.zeros((0, 3)),
            best_mask=np.zeros((0, 0), dtype=bool),
            n_frames_used=0,
        )

    vols = np.array([v for _, v, _ in frame_data], dtype=np.float64)

    # Tukey 1.5×IQR outlier rejection
    keep_mask = np.ones(len(vols), dtype=bool)
    if len(vols) >= 4:
        q1, q3 = np.percentile(vols, [25, 75])
        iqr = q3 - q1
        if iqr > 0:
            keep_mask = (vols >= q1 - 1.5*iqr) & (vols <= q3 + 1.5*iqr)
            if not keep_mask.any():
                keep_mask = np.ones(len(vols), dtype=bool)

    kept = [frame_data[i] for i in range(len(frame_data)) if keep_mask[i]]
    dropped = len(frame_data) - len(kept)
    if dropped:
        drop_str = [round(frame_data[i][1]*1000, 1) for i in range(len(frame_data)) if not keep_mask[i]]
        kept_str = [round(v*1000, 1) for _, v, _ in kept]
        print(f'    [iqr]  dropped {dropped}: {drop_str}L  kept: {kept_str}L')

    kept_vols = np.array([v for _, v, _ in kept], dtype=np.float64)
    _, _, best_result = max(kept, key=lambda x: x[0])

    all_pts = [r.point_cloud for _, _, r in kept if r.point_cloud.shape[0] > 0]
    merged_vol = 0.0
    if all_pts:
        merged_pts = np.concatenate(all_pts, axis=0)
        v_raw, _   = _compute_2d5_volume(merged_pts)
        merged_vol = v_raw * SCALE_2D5 if v_raw > 0 else 0.0

    return _pp_mod.BundleResult(
        median_volume=float(np.median(kept_vols)),
        std_volume=float(np.std(kept_vols)),
        mean_volume=float(np.mean(kept_vols)),
        merged_volume=merged_vol,
        per_frame_volumes=[v for _, v, _ in frame_data],
        best_point_cloud=best_result.point_cloud,
        best_mask=best_result.mask,
        n_frames_used=len(frame_data),
    )

_pp_mod.process_bundle_multi_frame = _patched_process_bundle
print('process_bundle_multi_frame patched (IQR outlier rejection).')

In [ ]:
# ── 4. Pipeline factory ────────────────────────────────────────────────────
import threading
import torch.nn as _nn

# torch.jit.script caches class types globally; concurrent init of two SAM2
# predictors causes "Can't redefine method: forward on Resize". Serialize.
_init_lock = threading.Lock()

if not getattr(_nn.Embedding.forward, '_device_safe_patched', False):
    _orig_emb_fwd = _nn.Embedding.forward

    def _device_safe_emb_fwd(self, input):
        if input.device != self.weight.device:
            input = input.to(self.weight.device)
        return _orig_emb_fwd(self, input)

    _device_safe_emb_fwd._device_safe_patched = True
    _nn.Embedding.forward = _device_safe_emb_fwd
    print('  nn.Embedding.forward patched (auto device placement)')
else:
    print('  nn.Embedding.forward already patched — skipping')

PIPELINE_KWARGS = dict(
    yolo_weights='yolov8m-world.pt',
    sam2_config='sam2_hiera_small.yaml',
    sam2_checkpoint='sam2_hiera_small.pt',
    use_depth_fusion=False,
    da_encoder='vits',
)

YOLO_CLASSES = ["fruit bunch"]


def make_pipeline(device):
    import torch as _t

    with _init_lock:
        p = FFBPerceptionPipeline(**PIPELINE_KWARGS, device=device)

    p.yolo.set_classes(YOLO_CLASSES)

    for _attr in ('txt_feats', 'text_feats'):
        for _obj in (p.yolo.model, getattr(p.yolo.model, 'model', None)):
            if _obj is None:
                continue
            _feat = getattr(_obj, _attr, None)
            if isinstance(_feat, _t.Tensor) and _feat.device.type == 'cpu':
                try:
                    setattr(_obj, _attr, _feat.to(device))
                except Exception:
                    pass

    print(f'  Pipeline ready on {device}  (YOLO classes: {YOLO_CLASSES})')
    return p

In [ ]:
# ── 5. Process all bundles (parallel on 2×T4, sequential otherwise) ───────
import traceback
from concurrent.futures import ThreadPoolExecutor

N_FRAMES = 15
MAX_SCAN = 120

def process_group(ffb_list, device):
    pl = make_pipeline(device)
    results = {}
    for ffb in ffb_list:
        ffb_dir = os.path.join(DATA_DIR, ffb)
        print(f'[{device}] {ffb} ...')
        try:
            depth_bag, _ = find_bundle_bags(ffb_dir)
            K = get_intrinsics_from_bag(depth_bag) or CameraIntrinsics.default_848x480()
            K.depth_scale = get_depth_scale(depth_bag)
            br = process_bundle_multi_frame(ffb_dir, pl, K,
                                            n_frames=N_FRAMES, max_scan=MAX_SCAN)
            results[ffb] = br
            g = gt(ffb)
            cv = 100 * br.std_volume / (br.median_volume + 1e-9)
            print(f'[{device}] {ffb}  hull={br.median_volume*1000:.2f} L  '
                  f'actual={g.get("Actual_Volume_L","?")} L  CV={cv:.1f}%')
        except Exception as e:
            print(f'[{device}] {ffb} ERROR: {e}')
            traceback.print_exc()
    return results

bundle_results = {}
if N_GPUS >= 2:
    mid = len(bundles) // 2
    print(f'Parallel: GPU0={bundles[:mid]}  GPU1={bundles[mid:]}')
    with ThreadPoolExecutor(max_workers=2) as pool:
        f0 = pool.submit(process_group, bundles[:mid], 'cuda:0')
        f1 = pool.submit(process_group, bundles[mid:], 'cuda:1')
        bundle_results.update(f0.result())
        bundle_results.update(f1.result())
else:
    dev = 'cuda:0' if N_GPUS == 1 else 'cpu'
    bundle_results.update(process_group(bundles, dev))

print(f'\nDone: {len(bundle_results)}/{len(bundles)} bundles.')

In [ ]:
# ── 5b. Calibrate SCALE_BY_WIDTH from ground truth ────────────────────────
# Derives the per-camera hemisphere scale factor from actual water-displacement
# volumes. Update SCALE_BY_WIDTH in Cell 3 if the ratios differ significantly.

_calib_rows = []

for ffb_name, br in bundle_results.items():
    g = gt(ffb_name)
    actual_vol_L = g.get('Actual_Volume_L', None)
    if actual_vol_L is None:
        continue
    actual_vol_m3 = float(actual_vol_L) / 1000.0

    v_raw, _ = _compute_2d5_volume(br.best_point_cloud)
    if v_raw <= 0:
        continue

    _calib_rows.append({
        'ffb':           ffb_name,
        'actual_vol_m3': actual_vol_m3,
        'v_2d5_raw':     v_raw,
        'ratio':         actual_vol_m3 / v_raw,
    })

if _calib_rows:
    _df_cal = pd.DataFrame(_calib_rows)
    _scale_cal = float(_df_cal['ratio'].mean())
    _scale_med = float(_df_cal['ratio'].median())
    print(f'Calibrated SCALE_2D5:  mean={_scale_cal:.3f}  median={_scale_med:.3f}')
    print(_df_cal[['ffb', 'actual_vol_m3', 'v_2d5_raw', 'ratio']].to_string(index=False))
    print(f'\nCurrent SCALE_BY_WIDTH: {SCALE_BY_WIDTH}')
    print('Update Cell 3 if ratios differ significantly.')
else:
    print('No FFBs with Actual_Volume_L in ground truth — skipping calibration.')

In [ ]:
# ── 6. Visual QC — mask overlays ──────────────────────────────────────────
from bag_reader import load_best_frame

n = len(bundle_results)
cols = min(n, 4); rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 5*rows))
axes = np.array(axes).flatten()

for ax, (ffb, br) in zip(axes, bundle_results.items()):
    rgb, _ = load_best_frame(os.path.join(DATA_DIR, ffb))
    mask_vis = cv2.resize(br.best_mask.astype(np.uint8),
                          (rgb.shape[1], rgb.shape[0]),
                          interpolation=cv2.INTER_NEAREST).astype(bool)
    overlay = rgb.copy()
    overlay[~mask_vis] = (overlay[~mask_vis] * 0.3).astype(np.uint8)
    g = gt(ffb)
    cv = 100 * br.std_volume / (br.median_volume + 1e-9)
    ax.imshow(overlay)
    ax.set_title(f"{ffb}  {g.get('Actual_Mass_kg','?')} kg\n"
                 f"{br.median_volume*1000:.2f} L  CV={cv:.1f}%", fontsize=7)
    ax.axis('off')
for ax in axes[n:]: ax.axis('off')
plt.tight_layout(); plt.savefig('mask_qc.png', dpi=150); plt.show()

In [ ]:
# ── 7. Results table ──────────────────────────────────────────────────────
rows = []
for ffb, br in bundle_results.items():
    g        = gt(ffb)
    hull_L   = br.median_volume * 1000
    hull_m3  = br.median_volume
    rows.append({
        'bundle':              ffb,
        'hull_vol_L':          round(hull_L, 3),
        'hull_cv_pct':         round(100*br.std_volume/(br.median_volume+1e-9), 2),
        'actual_vol_L':        g.get('Actual_Volume_L',  float('nan')),
        'vol_error_L':         round(hull_L - g.get('Actual_Volume_L', hull_L), 3),
        'actual_mass_kg':      g.get('Actual_Mass_kg',   float('nan')),
        'true_density_kg_L':   g.get('True_Density_kg_L', float('nan')),
        'pred_mass_956':       round(DENSITY_CONSTANT * hull_m3, 2),
        'pred_mass_oracle':    round(g.get('True_Density_kg_L', float('nan')) * 1000 * hull_m3, 2),
        'n_frames':            br.n_frames_used,
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv('ffb_pipeline_results.csv', index=False)
print(f'\nDensity constant: {DENSITY_CONSTANT:.2f} kg/m³')

In [ ]:
# ── 8. Evaluation ─────────────────────────────────────────────────────────
from scipy.stats import pearsonr

df_eval = df.dropna(subset=['actual_mass_kg']).copy()
y_true  = df_eval['actual_mass_kg'].values

def metrics(y_true, y_pred, label):
    mae   = np.mean(np.abs(y_pred - y_true))
    mape  = np.mean(np.abs(y_pred - y_true) / y_true) * 100
    rmse  = np.sqrt(np.mean((y_pred - y_true) ** 2))
    r_p, _= pearsonr(y_true, y_pred)
    r2    = 1 - np.sum((y_true - y_pred)**2) / (np.sum((y_true - y_true.mean())**2) + 1e-9)
    print(f'  {label}')
    print(f'    MAE={mae:.2f} kg  MAPE={mape:.1f}%  RMSE={rmse:.2f} kg  '
          f'r={r_p:.4f}  r²={r2:.4f}')
    return mae, mape, rmse, r_p, r2

print('=== Mass prediction ===')
m1 = metrics(y_true, df_eval['pred_mass_956'].values,
             f'2.5D vol × {DENSITY_CONSTANT:.2f} kg/m³  (SCALE_2D5={SCALE_2D5})')
m2 = metrics(y_true, df_eval['pred_mass_oracle'].values,
             'Vol × per-bunch true density  [oracle upper bound]')

# ── Aqil baseline reference ───────────────────────────────────────────────
print('\n=== Aqil thesis baseline (CloudCompare 2.5D, manual seg, n=50) ===')
print('  Volume:  r=0.9623  r²=0.9261')
print('  Mass:    r=0.9496  r²=0.9019')
print('  Validation (n=10):  r=0.9503  r²=0.9032')

# ── Volume correlation ────────────────────────────────────────────────────
df_vol = df_eval.dropna(subset=['actual_vol_L']).copy()
if len(df_vol) >= 2:
    print('\n=== Volume estimation ===')
    v_actual = df_vol['actual_vol_L'].values
    v_pred   = df_vol['hull_vol_L'].values
    r_v, _   = pearsonr(v_actual, v_pred)
    r2_v     = 1 - np.sum((v_actual - v_pred)**2) / (np.sum((v_actual - v_actual.mean())**2) + 1e-9)
    mae_v    = np.mean(np.abs(v_pred - v_actual))
    print(f'  MAE={mae_v:.2f} L  r={r_v:.4f}  r²={r2_v:.4f}  (vs Aqil: r=0.9623  r²=0.9261)')

# ── Plots ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Mass scatter
ax = axes[0]
ax.scatter(y_true, df_eval['pred_mass_956'],
           label=f'956 kg/m³  r={m1[3]:.3f}  r²={m1[4]:.3f}', s=60)
ax.scatter(y_true, df_eval['pred_mass_oracle'],
           label=f'Oracle      r={m2[3]:.3f}  r²={m2[4]:.3f}', marker='^', s=60)
for _, row in df_eval.iterrows():
    ax.annotate(row['bundle'], (row['actual_mass_kg'], row['pred_mass_956']),
                fontsize=7, xytext=(3, 3), textcoords='offset points')
lim = [y_true.min() * 0.85, y_true.max() * 1.1]
ax.plot(lim, lim, 'r--', alpha=0.4, label='1:1')
# Regression line
m_coef = np.polyfit(y_true, df_eval['pred_mass_956'].values, 1)
x_fit  = np.linspace(*lim, 50)
ax.plot(x_fit, np.polyval(m_coef, x_fit), 'b:', alpha=0.6,
        label=f'fit y={m_coef[0]:.3f}x+{m_coef[1]:.2f}')
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel('Actual mass (kg)'); ax.set_ylabel('Predicted mass (kg)')
ax.set_title('Mass prediction'); ax.legend(fontsize=7)

# Volume scatter
ax = axes[1]
if len(df_vol) >= 2:
    ax.scatter(v_actual, v_pred, s=60)
    for _, row in df_vol.iterrows():
        ax.annotate(row['bundle'], (row['actual_vol_L'], row['hull_vol_L']),
                    fontsize=7, xytext=(3, 3), textcoords='offset points')
    vmax = max(v_actual.max(), v_pred.max()) * 1.1
    ax.plot([0, vmax], [0, vmax], 'r--', alpha=0.4, label='1:1')
    v_coef = np.polyfit(v_actual, v_pred, 1)
    ax.plot(np.linspace(0, vmax, 50),
            np.polyval(v_coef, np.linspace(0, vmax, 50)), 'b:',
            label=f'fit y={v_coef[0]:.3f}x+{v_coef[1]:.3f}')
    ax.set_xlabel('Water-displacement vol (L)'); ax.set_ylabel('Estimated vol (L)')
    ax.set_title(f'Volume  r={r_v:.3f}  r²={r2_v:.3f}\n(Aqil: r=0.9623  r²=0.9261)')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('evaluation.png', dpi=150)
plt.show()

In [ ]:
# ── 9. 3D point cloud visualisation ───────────────────────────────────────
def show_pc(pts, title):
    try:
        import open3d as o3d
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(pts)
        z = (pts[:,2]-pts[:,2].min()) / (pts[:,2].max()-pts[:,2].min()+1e-6)
        pcd.colors = o3d.utility.Vector3dVector(plt.cm.viridis(z)[:,:3])
        o3d.io.write_point_cloud(f'{title}.ply', pcd)
        vis = o3d.visualization.Visualizer()
        vis.create_window(visible=False, width=800, height=600)
        vis.add_geometry(pcd); vis.poll_events(); vis.update_renderer()
        vis.capture_screen_image(f'{title}_pc.png'); vis.destroy_window()
        plt.figure(figsize=(5,4)); plt.imshow(plt.imread(f'{title}_pc.png'))
        plt.axis('off'); plt.title(title); plt.show()
    except Exception:
        idx = np.random.choice(len(pts), min(5000, len(pts)), replace=False)
        s = pts[idx]
        fig = plt.figure(figsize=(5,4))
        ax3 = fig.add_subplot(111, projection='3d')
        ax3.scatter(s[:,0], s[:,1], s[:,2], c=s[:,2], cmap='viridis', s=1)
        ax3.set_title(title); plt.tight_layout(); plt.show()

for ffb, br in list(bundle_results.items())[:3]:
    show_pc(br.best_point_cloud, ffb)

In [ ]:
# ── 10. Save point clouds for downstream training ─────────────────────────
os.makedirs('point_clouds', exist_ok=True)
for ffb, br in bundle_results.items():
    g = gt(ffb)
    np.savez_compressed(
        f'point_clouds/{ffb}.npz',
        point_cloud=br.best_point_cloud,
        mask=br.best_mask,
        median_volume=np.array(br.median_volume),
        std_volume=np.array(br.std_volume),
        per_frame_volumes=np.array(br.per_frame_volumes),
        actual_mass_kg=np.array(g.get('Actual_Mass_kg',   float('nan'))),
        actual_volume_L=np.array(g.get('Actual_Volume_L', float('nan'))),
        true_density=np.array(g.get('True_Density_kg_L',  float('nan'))),
    )
    print(f'{ffb}: {br.best_point_cloud.shape}  '
          f'mass={g.get("Actual_Mass_kg","?")} kg  '
          f'hull={br.median_volume*1000:.2f} L')
print('Done.')

## Metrics
| | |
|--|--|
| **MAE** | mean absolute error in kg |
| **MAPE** | mean absolute % error — fair across bunch sizes |
| **RMSE** | penalises large outliers |
| **R²** | 1=perfect, 0=no better than predicting the mean |

**956.28 kg/m³ baseline** — hull volume × population-mean density, validated on Aqil's 10 independent points.  
**Oracle** — hull volume × per-bunch true density (upper bound; not achievable without knowing density in advance).